In [ ]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# Enable autoreload for development (automatically reloads modules when they change)
%load_ext autoreload
%autoreload 2

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Find libnvrtc.so.12 and preload it using ctypes
    # This ensures CuPy can find it even if LD_LIBRARY_PATH isn't fully respected
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                
                # Preload the library using ctypes so CuPy can find it
                # Use RTLD_GLOBAL to make symbols available to other libraries
                try:
                    import ctypes
                    # Try multiple loading strategies
                    try:
                        # Strategy 1: Load with full path and RTLD_GLOBAL
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        # Strategy 2: Try without RTLD_GLOBAL
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                    print(f"  CuPy may still work if LD_LIBRARY_PATH is set correctly")
                    print(f"  You may need to restart the Jupyter kernel with:")
                    print(f"    export LD_LIBRARY_PATH={os.path.dirname(libnvrtc_path)}:$LD_LIBRARY_PATH")
                
                # Verify that ctypes can find the library by name (as CuPy will try)
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                    else:
                        print(f"⚠ ctypes.util.find_library('nvrtc') returned None")
                        print(f"  This may cause issues. Try loading by name:")
                        try:
                            test_lib = ctypes.CDLL('libnvrtc.so.12')
                            print(f"✓ Successfully loaded libnvrtc.so.12 by name")
                        except Exception as name_err:
                            print(f"✗ Failed to load libnvrtc.so.12 by name: {name_err}")
                            print(f"  You MUST restart the Jupyter kernel with LD_LIBRARY_PATH set")
                except Exception as diag_err:
                    print(f"⚠ Could not run diagnostics: {diag_err}")
                break
    
    if not libnvrtc_path:
        # Try to find any version of libnvrtc.so
        import glob
        for cuda_lib_path in cuda_lib_paths:
            if os.path.exists(cuda_lib_path):
                nvrtc_files = glob.glob(os.path.join(cuda_lib_path, 'libnvrtc.so*'))
                if nvrtc_files:
                    # Try to use the most specific version
                    nvrtc_files.sort(reverse=True)  # Prefer .so.12.2.140 over .so.12 over .so
                    potential_lib = nvrtc_files[0]
                    print(f"⚠ libnvrtc.so.12 not found, but found: {nvrtc_files}")
                    print(f"  Attempting to use: {potential_lib}")
                    try:
                        import ctypes
                        ctypes.CDLL(potential_lib, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded {potential_lib} using ctypes")
                        libnvrtc_path = potential_lib
                    except Exception as e:
                        print(f"⚠ Could not preload {potential_lib}: {e}")
                    break
        else:
            print(f"⚠ Warning: libnvrtc.so.12 not found in any CUDA library directory")
            print(f"  This may cause CuPy kernel compilation to fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")

"""
Verification tests for H function (observation probability) in BeliefMDP_n.

Tests:
1. Batched vs sequential H computation performance
2. Vectorized H methods comparison
3. H normalization verification
4. H vs H_log performance comparison
5. Q density unit integration

Uses the same model as T_mat_visuals.ipynb:
- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=8)
"""
# Use CuPy backend for GPU acceleration (falls back to NumPy if not available)
from src.utils.array_backend import np, random, is_cupy
from src.belief_quantized.belief_mdp_n import BeliefMDP_n_SLAM as BeliefMDP_n
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Verification tests for H function (observation probability) in BeliefMDP_n.

Tests:
1. Batched vs sequential H computation performance
2. Vectorized H methods comparison
3. H normalization verification
4. H vs H_log performance comparison
5. Q density unit integration

Uses the same model as T_mat_visuals.ipynb:
- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=8)
"""


CWD: /global/home/hpc5656/SLAM
CUDA_PATH not set, attempting to load modules...
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH to include: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64, /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/targets/x86_64-linux/lib
✓ Found libnvrtc.so.12 at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libnvrtc.so.12
✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)
✓ ctypes.util.find_library('nvrtc') found: libnvrtc.so.12
✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
✓ All imports successful
Using backend: CuPy (GPU)


In [ ]:
def test_Q_density_unit_integration():
    """
    Test 1: Verify that Q returns probability DENSITY (not measure).

    For a proper density function q(y|x,m), we should have:
    ∫ q(y|x,m) dy = 1

    Since observations are continuous, we can numerically verify this.
    """
    obstacles, area = load_obstacles_config(environment='toy2')
    # Use same model as T_mat_visuals.ipynb
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=2,  # Match n=4 from T_mat_visuals
    )
    bmdp = BeliefMDP_n(
        n=2,  # Match T_mat_visuals 
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)

    # Pick a test state and map (use middle state, but adjust index for 4D state space)
    # For 4D states, we need to pick a reasonable index
    mid_idx = bmdp.SQ.m_n // 2
    x_test = bmdp.SQ.X_n[mid_idx]  # Middle state (4D: [x, y, vx, vy])
    m_test = bmdp.map.occupancy_map.data  # True map

    # Sample observation space Y = [0, r_max]^B
    B = sensor.B
    r_max = sensor.r_max
    n_samples_per_dim = 50

    # Create grid in observation space
    y_values = np.linspace(0.1, r_max, n_samples_per_dim)
    total_integral = 0.0
    dy_volume = (r_max / n_samples_per_dim) ** B  # Volume element

    print(f"\n=== Testing Q density integration ===")
    print(f"State: {x_test}, Map shape: {m_test.shape}")
    print(f"Observation space: [0, {r_max}]^{B}")
    print(f"Sampling grid: {n_samples_per_dim}^{B} points")

    # DIAGNOSTICS: Check what's happening
    print(f"\n=== Diagnostics ===")
    print(f"sigma_v (observation noise): {bmdp.σ_v}")
    print(f"cov_y shape: {bmdp.cov_y.shape}")
    print(f"cov_y determinant: {np.linalg.det(bmdp.cov_y)}")
    print(f"cov_y diagonal (first few): {np.diag(bmdp.cov_y)[:5]}")
    
    # Get ideal observation for the test state
    x_test_pos = x_test[:2]  # Position only
    y_star_test = bmdp.ray_casting(x_test_pos[np.newaxis, :], m_test)[0]
    print(f"\nIdeal observation y_star for test state: {y_star_test}")
    print(f"y_star range: [{np.min(y_star_test):.3f}, {np.max(y_star_test):.3f}]")
    
    # Check a few sample observations
    print(f"\nSample observations vs y_star:")
    for i in range(3):
        y_sample = random.uniform(0.1, r_max, B)
        diff = y_sample - y_star_test
        diff_norm_sq = np.dot(diff, diff)
        print(f"  Sample {i}: y={y_sample[:3]}..., ||y-y_star||²={diff_norm_sq:.3f}")
    
    # Numerically integrate Q over observation space
    Q_values = []
    y_star_ideal = bmdp.ray_casting(x_test_pos[np.newaxis, :], m_test)[0]
    
    for i in tqdm(range(100), desc="Sampling Q values"):  # Sample 100 random points
        y_sample = random.uniform(0.1, r_max, B)
        q_val = bmdp.Q(y_sample, x_test[np.newaxis, :], m_test)[0]
        # Convert CuPy/NumPy scalar to Python float if needed
        q_val_float = float(q_val.item()) if hasattr(q_val, 'item') else float(q_val)
        Q_values.append(q_val_float)
        
        # Diagnostic for first few samples
        if i < 3:
            diff = y_sample - y_star_ideal
            diff_norm_sq = np.dot(diff, diff)
            print(f"  Sample {i}: Q={q_val_float:.2e}, ||y-y_star||²={diff_norm_sq:.3f}")

    Q_values_arr = np.array(Q_values)
    print(f"\n=== Results ===")
    print(f"Mean Q(y|x,m): {np.mean(Q_values_arr):.6e}")
    print(f"Max Q(y|x,m): {np.max(Q_values_arr):.6e}")
    print(f"Min Q(y|x,m): {np.min(Q_values_arr):.6e}")
    print(f"Number of zero values: {np.sum(Q_values_arr == 0)}/{len(Q_values_arr)}")
    print(f"Number of non-zero values: {np.sum(Q_values_arr > 0)}/{len(Q_values_arr)}")

    # For multivariate normal: Q(y|x,m) = N(y; g_bar(x,m), Σ_v)
    # The integral over ℝ^B should equal 1
    # We expect some variation due to numerical sampling
    assert np.all(Q_values_arr > 0), "Q should return positive densities"
    assert np.all(Q_values_arr < np.inf), "Q should return finite densities"

    print("✓ Q returns valid probability density values")
    
test_Q_density_unit_integration()
